In [ ]:
# 2D Molecular Dynamics with embedded animation (Velocity-Verlet, Lennard-Jones)
# This code will run a short MD simulation and display an inline animation using matplotlib.
# It uses reduced LJ units (epsilon = sigma = mass = 1).
# The simulation includes periodic boundaries and a simple neighbor cutoff (no cell lists for clarity).
# Adjust N, steps, dt etc. as desired.

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from matplotlib import rc
rc('animation', embed_limit = 50)

np.random.seed(1)

# --- Parameters ---
N = 36                    # number of particles
rho = 0.4                 # number density (N / L^2)
L = np.sqrt(N / rho)      # box length
dt = 0.01                # time step (reduced units)
steps = 2000              # total steps to integrate
frames = 1000              # frames in the animation
frame_interval = 2  # integrate this many steps between animation frames
rc = 2.5                  # LJ cutoff (in sigma units)
rc2 = rc**2

# LJ parameters (reduced)
epsilon = 1.0
sigma = 1.0
mass = 1.0

# --- Helper functions ---
def minimum_image(rij, L):
    """Apply minimum image convention for displacement vector rij in box L."""
    return rij - L * np.round(rij / L)

def compute_forces(positions):
    """Compute forces and potential energy using Lennard-Jones potential with cutoff."""
    forces = np.zeros_like(positions)
    U = 0.0
    for i in range(N-1):
        for j in range(i+1, N):
            rij = positions[i] - positions[j]
            rij = minimum_image(rij, L)
            r2 = np.dot(rij, rij)
            if r2 < rc2:
                invr2 = 1.0 / r2
                invr6 = invr2**3
                # LJ potential: 4*epsilon*( (sigma/r)^12 - (sigma/r)^6 )
                # Force magnitude (scalar) = 24*epsilon*invr2 * (2*invr6**2 - invr6)
                fscalar = 24 * epsilon * invr2 * (2 * invr6**2 - invr6)
                fij = fscalar * rij
                forces[i] += fij
                forces[j] -= fij
                U += 4 * epsilon * (invr6**2 - invr6) - U_shift
    return forces, U

def apply_periodic(positions):
    """Wrap positions into the periodic box [0,L)."""
    return positions % L

# Precompute potential shift so U(rc)=0 (shifted LJ)
invr2_rc = 1.0 / rc2
invr6_rc = invr2_rc**3
U_shift = 4 * epsilon * (invr6_rc**2 - invr6_rc)

# --- Initialize positions (small square lattice) and velocities (Maxwell-Boltzmann-like) ---
n_side = int(np.ceil(np.sqrt(N)))
grid = np.linspace(0, L, n_side+1)[:-1]
xs, ys = np.meshgrid(grid, grid)
pos = np.vstack([xs.ravel(), ys.ravel()]).T[:N].astype(float)

# small random displacement to break symmetry
pos += 0.2 * (np.random.rand(*pos.shape) - 0.5)

# velocities: random then zero total momentum
v = np.random.normal(0, 1.0, size=(N,2))
v -= v.mean(axis=0)

# scale velocities to target temperature ~1.0 (in reduced units)
kinetic = 0.5 * mass * np.sum(v**2)
T_current = (2.0 * kinetic) / (2.0 * N)  # 2D
scale = np.sqrt(1.0 / T_current)
v *= 1.8

# Compute initial forces
forces, U = compute_forces(pos)

# For logging energies
energies = []
times = []

# --- Integrator: Velocity-Verlet ---
def integrate_steps(positions, velocities, forces, nsteps):
    """Integrate nsteps steps of velocity-Verlet and return final positions, velocities, forces, and energies list."""
    global U_shift
    local_U = 0.0
    pos = positions.copy()
    vel = velocities.copy()
    f = forces.copy()
    for _ in range(nsteps):
        # half step velocity
        vel += 0.5 * f / mass * dt
        # full step position
        pos += vel * dt
        pos = apply_periodic(pos)
        # compute new forces
        f_new, U_new = compute_forces(pos)
        # half step velocity
        vel += 0.5 * f_new / mass * dt
        f = f_new
        local_U = U_new
    return pos, vel, f, local_U

# --- Run short simulation while collecting frames for animation ---
positions_history = []
pos_now = pos.copy()
v_now = v.copy()
f_now = forces.copy()
U_now = U

# Storage
kinetic_list = []
potential_list = []
total_list = []
times = []

for frame in range(frames):
    pos_now, v_now, f_now, U_now = integrate_steps(pos_now, v_now, f_now, frame_interval)

    positions_history.append(pos_now.copy())

    # --- compute kinetic energy ---
    K = 0.5 * mass * np.sum(v_now**2)

    # Store energies
    kinetic_list.append(K)
    potential_list.append(U_now)
    total_list.append(K + U_now)

    # Time array
    times.append((frame+1) * frame_interval * dt)


# --- Build animation ---
fig, ax = plt.subplots(figsize=(6,6))
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(0, L)
ax.set_ylim(0, L)
ax.set_title('2D Molecular Dynamics — Lennard-Jones (reduced units)')
scat = ax.scatter([], [])

# draw box boundary
ax.plot([0, L, L, 0, 0], [0, 0, L, L, 0])

def init():
    scat.set_offsets(np.empty((0,2)))
    return (scat,)

def update(frame):
    coords = positions_history[frame]
    scat.set_offsets(coords)
    ax.set_xlabel(f'Frame {frame+1}/{frames}  Time {times[frame]:.3f}')
    return (scat,)

anim = animation.FuncAnimation(fig, update, frames=len(positions_history), init_func=init,
                               blit=True, interval=30)

# Display the animation inline
html_anim = HTML(anim.to_jshtml())

# Also show an energy plot below
fig2, ax2 = plt.subplots(figsize=(6,2.5))
ax2.plot(times, total_list)
ax2.set_xlabel('time')
ax2.set_ylabel('total energy')
ax2.set_title('Total energy vs time')
plt.show()

# Display both the animation (HTML) and the energy plot image
display(html_anim)
plt.show()

# Return the animation HTML so the notebook / front-end shows it as output of the cell
html_anim._repr_html_()




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------- Minimum-image function --------
def minimum_image(vec, L):
    return vec - L * np.round(vec / L)

# -------- Compute MSD --------
pos0 = positions_history[0]         # initial positions
N = pos0.shape[0]

msd = []
for pos in positions_history:
    disp = pos - pos0
    disp = minimum_image(disp, L)   # periodic boundary correction
    msd.append(np.mean(np.sum(disp**2, axis=1)))

msd = np.array(msd)

# -------- Choose x-axis (use the one your assignment expects) --------
# Option A: time axis (if you stored "times")
x = np.array(times)
x_label = "Time"

# Option B: frame number (uncomment these two lines if needed)
# x = np.arange(len(msd))
# x_label = "Frame number"

# -------- Plot (styled like your example) --------
plt.figure(figsize=(6,4))
plt.plot(x, msd, color="purple", linewidth=2)   # purple line like screenshot
plt.title("Mean-Squared Displacement")
plt.xlabel(x_label)
plt.ylabel("MSD")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4))

plt.plot(times, kinetic_list, label="Kinetic Energy", linewidth=2)
plt.plot(times, potential_list, label="Potential Energy", linewidth=2)
plt.plot(times, total_list, label="Total Energy", linewidth=2)

plt.xlabel("Time")
plt.ylabel("Energy")
plt.title("Energy vs Time")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Vectorized 2D MD (Velocity-Verlet) with MSD + energy plots + safe inline animation
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

np.random.seed(1)

# ---------------- Parameters ----------------
N = 36                   # number of particles
rho = 0.4                # number density
L = np.sqrt(N / rho)     # box length
dt = 0.01                # timestep
steps = 2000             # total integration steps
save_interval = 4        # save positions every this many steps (reduce to limit animation size)
rc = 2.5
rc2 = rc**2

epsilon = 1.0
sigma = 1.0
mass = 1.0

# ---------------- Helper functions ----------------
def minimum_image_vec(dR, L):
    """Apply minimum image to an array of displacement vectors dR with shape (...,2)."""
    return dR - L * np.round(dR / L)

def compute_forces_and_potential(pos):
    """
    Vectorized force and potential calculation.
    pos: (N,2)
    Returns: forces (N,2), potential_energy (scalar)
    """
    # pairwise displacement arrays: rij[i,j,:] = r_i - r_j
    rij = pos[:, None, :] - pos[None, :, :]            # shape (N,N,2)
    rij = minimum_image_vec(rij, L)
    r2 = np.sum(rij**2, axis=2)                       # shape (N,N)
    # avoid self-interaction
    np.fill_diagonal(r2, np.inf)
    mask = r2 < rc2

    invr2 = np.zeros_like(r2)
    invr2[mask] = 1.0 / r2[mask]
    invr6 = invr2**3
    # pairwise scalar force magnitude divided by r (so we can multiply by rij)
    # fscalar = 24*epsilon*invr2 * (2*invr6**2 - invr6)  <--- already 1/r factor included
    fscalar = np.zeros_like(r2)
    fscalar[mask] = 24 * epsilon * invr2[mask] * (2 * invr6[mask]**2 - invr6[mask])
    # force vectors: sum_j fscalar[i,j] * rij[i,j,:]
    forces = np.einsum('ij,ijk->ik', fscalar, rij)

    # potential energy: sum over pairs 4*(inv6^2 - inv6) - U_shift, but only count each pair once
    invr6_masked = invr6 * mask
    pair_pot = 4 * epsilon * (invr6_masked**2 - invr6_masked)
    # shift so U(rc)=0
    invr2_rc = 1.0 / rc2
    invr6_rc = invr2_rc**3
    U_shift = 4 * epsilon * (invr6_rc**2 - invr6_rc)
    pair_pot = pair_pot - U_shift * mask
    # sum over i<j: half of sum over all pairs since matrix is symmetric
    U = 0.5 * np.sum(pair_pot)

    return forces, U

def apply_periodic(positions):
    return positions % L

# ---------------- Initialization ----------------
n_side = int(np.ceil(np.sqrt(N)))
grid = np.linspace(0, L, n_side+1)[:-1]
xs, ys = np.meshgrid(grid, grid)
pos = np.vstack([xs.ravel(), ys.ravel()]).T[:N].astype(float)
pos += 0.2 * (np.random.rand(*pos.shape) - 0.5)   # small perturbation

v = np.random.normal(0, 1.0, size=(N,2))
v -= v.mean(axis=0)
# scale velocities (you used v *= 1.8 earlier; keep that if you want higher initial kinetic)
# compute current temperature then scale to target T=1
kinetic0 = 0.5 * mass * np.sum(v**2)
T_current = (2.0 * kinetic0) / (2.0 * N)  # 2D
v *= np.sqrt(1.0 / T_current) * 1.8      # keep your 1.8 multiplier from previous attempt

# compute initial forces & potential
forces, U = compute_forces_and_potential(pos)

# ---------------- Integrator (Velocity-Verlet) ----------------
positions_history = []   # saved snapshots
times_saved = []
kinetic_list = []
potential_list = []
total_list = []

pos_now = pos.copy()
v_now = v.copy()
f_now = forces.copy()
U_now = U

for step in range(steps):
    # Velocity-Verlet
    v_now += 0.5 * f_now / mass * dt
    pos_now += v_now * dt
    pos_now = apply_periodic(pos_now)
    f_new, U_new = compute_forces_and_potential(pos_now)
    v_now += 0.5 * f_new / mass * dt
    f_now = f_new
    U_now = U_new

    # record every save_interval steps (this keeps animation small)
    if (step % save_interval) == 0:
        positions_history.append(pos_now.copy())
        t = (step+1) * dt
        times_saved.append(t)
        K = 0.5 * mass * np.sum(v_now**2)
        kinetic_list.append(K)
        potential_list.append(U_now)
        total_list.append(K + U_now)

# convert lists to arrays
positions_history = np.array(positions_history)   # shape (frames, N, 2)
times_saved = np.array(times_saved)
kinetic_list = np.array(kinetic_list)
potential_list = np.array(potential_list)
total_list = np.array(total_list)

print(f"Saved {len(positions_history)} frames (save_interval={save_interval}).")

# ---------------- Mean-Squared Displacement (MSD) ----------------
# compute MSD using initial saved snapshot as reference
pos0 = positions_history[0]   # (N,2)
def minimum_image_displacements(cur_pos, ref_pos):
    d = cur_pos - ref_pos
    d = minimum_image_vec(d, L)
    return d

msd = []
for frame_pos in positions_history:
    d = minimum_image_displacements(frame_pos, pos0)   # (N,2)
    msd.append(np.mean(np.sum(d**2, axis=1)))
msd = np.array(msd)

# ---------------- Plotting ----------------
# 1) MSD plot — x-axis as sample/frame number like your sample (so it matches your provided picture)
x = np.arange(len(msd))               # frame/sample number axis
plt.figure(figsize=(6,4))
plt.plot(x, msd, linewidth=1.6)       # leave default color (you can set color="purple")
plt.title("Mean-Squared Displacement")
plt.xlabel("Frame number")            # matches your sample
plt.ylabel("MSD")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

# 2) Energies: kinetic, potential, total vs time (use saved times)
plt.figure(figsize=(7,4))
plt.plot(times_saved, kinetic_list, label="Kinetic")
plt.plot(times_saved, potential_list, label="Potential")
plt.plot(times_saved, total_list, label="Total")
plt.xlabel("Time")
plt.ylabel("Energy")
plt.title("Kinetic, Potential, and Total Energy vs time")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3) Optional inline animation (kept small by save_interval)
# If you want full 1:1 frames, set save_interval=1 but be careful the animation may exceed embed limits.
from matplotlib import animation
fig, ax = plt.subplots(figsize=(5,5))
ax.set_xlim(0, L)
ax.set_ylim(0, L)
ax.set_aspect('equal')
ax.plot([0, L, L, 0, 0],[0,0,L,L,0], color='k')
scat = ax.scatter([], [], s=40)

def init():
    scat.set_offsets(np.empty((0,2)))
    return (scat,)

def update(i):
    scat.set_offsets(positions_history[i])
    ax.set_title(f"Frame {i+1}/{len(positions_history)}  Time {times_saved[i]:.3f}")
    return (scat,)

anim = animation.FuncAnimation(fig, update, frames=len(positions_history),
                               init_func=init, blit=True, interval=40)

# Inline embed (may be large if you saved many frames). If too big, reduce save_interval or save to mp4.
try:
    html = HTML(anim.to_jshtml())
    display(html)
except Exception as e:
    print("Inline animation generation failed or exceeded size; you can save to mp4 instead.")
    print(e)
    # To save to mp4 (uncomment if ffmpeg installed in your env):
    # Writer = animation.writers['ffmpeg']
    # writer = Writer(fps=25, metadata=dict(artist='me'), bitrate=1800)
    # anim.save("md2d_animation.mp4", writer=writer)

In [ ]:
Nx = 7
Ny = 4
N = Nx * Ny * 2      # two atoms per hexagonal cell
L = Nx * 2**(1/6)    # equilibrium bond length

ix = np.repeat(np.arange(Nx), Ny*2)
iy = np.tile(np.repeat(np.arange(Ny), 2), Nx)
iz = np.tile(np.array([0, 1]), Nx*Ny)

# hexagonal basis
x = ix + 0.5 * iz
y = (iy + 0.5 * iz) * Nx/Ny

# scale to box size
pos = np.column_stack((x, y))
pos = pos / Nx * L   # normalize so 0..L


In [ ]:
np.random.seed(1)

Nx = 7
Ny = 4
N = Nx * Ny * 2                      # hexagonal has 2 atoms per unit cell
L = Nx * 2**(1.0/6.0)                # L chosen so L/Nx = LJ eq. length

dt = 0.01
steps = 2000
frames = 2000
frame_interval = 1

rc = 2.5
rc2 = rc * rc

epsilon = 1.0
sigma = 1.0
mass   = 1.0

def minimum_image(rij, L):
    return rij - L * np.round(rij / L)

def compute_forces(positions):
    forces = np.zeros_like(positions)
    U = 0.0
    for i in range(N-1):
        for j in range(i+1, N):
            rij = minimum_image(positions[i] - positions[j], L)
            r2  = np.dot(rij, rij)
            if r2 < rc2:
                invr2 = 1.0 / r2
                invr6 = invr2**3
                fmag  = 24*epsilon * invr2 * (2*invr6**2 - invr6)
                fij   = fmag * rij
                forces[i] += fij
                forces[j] -= fij
                U += 4*epsilon*(invr6**2 - invr6) - U_shift
    return forces, U

def apply_periodic(pos):
    return pos % L

# Precompute shift so U(rc)=0
invr2_rc = 1.0 / rc2
invr6_rc = invr2_rc**3
U_shift  = 4*epsilon*(invr6_rc**2 - invr6_rc)

pos = np.zeros((N, 2))
i = 0
for ix in range(Nx):
    for iy in range(Ny):
        for iz in range(2):
            x = ix + 0.5 * iz
            y = (iy + 0.5 * iz) * Nx / Ny
            pos[i, :] = np.array([x, y]) / Nx * L
            i += 1


v = np.random.normal(0, 1.0, size=(N, 2))
v -= v.mean(axis=0)          # zero momentum

# Optional temperature scaling
K = 0.5 * mass * np.sum(v*v)
T_current = (2*K) / (2*N)
v /= np.sqrt(T_current)      # scale to T=1

# Initial forces
forces, U = compute_forces(pos)

# Storage
kinetic_list   = []
potential_list = []
total_list     = []
times          = []


def integrate_steps(pos, vel, f, nsteps):
    for _ in range(nsteps):
        vel += 0.5 * f * dt
        pos += vel * dt
        pos = apply_periodic(pos)
        f_new, U_new = compute_forces(pos)
        vel += 0.5 * f_new * dt
        f = f_new
    return pos, vel, f, U_new


positions_history = []
pos_now = pos.copy()
v_now   = v.copy()
f_now   = forces.copy()
U_now   = U

for frame in range(frames):
    pos_now, v_now, f_now, U_now = integrate_steps(pos_now, v_now, f_now, frame_interval)

    positions_history.append(pos_now.copy())

    K = 0.5 * mass * np.sum(v_now**2)
    kinetic_list.append(K)
    potential_list.append(U_now)
    total_list.append(K + U_now)

    times.append((frame+1)*dt)


fig, ax = plt.subplots(figsize=(6,6))
ax.set_xlim(0, L)
ax.set_ylim(0, L)
ax.set_aspect("equal")

sc = ax.scatter([], [])

def init():
    sc.set_offsets(np.zeros((N,2)))
    return (sc,)

def update(i):
    sc.set_offsets(positions_history[i])
    return (sc,)

anim = animation.FuncAnimation(fig, update, frames=frames, init_func=init, blit=True)
display(HTML(anim.to_jshtml()))

plt.figure(figsize=(6,4))
plt.plot(times, kinetic_list, label="Kinetic")
plt.plot(times, potential_list, label="Potential")
plt.plot(times, total_list, label="Total")
plt.legend()
plt.xlabel("time")
plt.ylabel("Energy")
plt.title("Energy vs Time")
plt.show()


In [ ]:
T_int = 2
N = 56
def Temp(vel,N):
    s = 0.0
    for i in range(N):
        s += vel[i,0]**2 + vel[i,1]**2
    T = s/2/(N-1)
    return T
vel = np.random.normal(0.0, np.sqrt(T_int), size = (N,2))
vel -= vel.mean(axis = 0)
print(T_int, Temp(vel,N))
vel *= np.sqrt(T_int / Temp(vel,N))
print(T_int, Temp(vel,N))

In [ ]:
def compute_msd(positions_history, L):
    """Return MSD(t) averaged over all particles."""
    pos0 = positions_history[0]  # reference frame
    msd = []
    for pos in positions_history:
        dr = pos - pos0
        dr -= L * np.round(dr / L)   # minimum image (periodic BC)
        msd.append(np.mean(np.sum(dr**2, axis=1)))
    return np.array(msd)

In [ ]:
Nx = 7      # small cell example (N = 56)
Ny = 4
N = Nx * Ny * 2

L = Nx * 2**(1/6)   # choose L so that L/Nx = equilibrium LJ spacing

pos = np.zeros((N, 2))

i = 0
for ix in range(Nx):
    for iy in range(Ny):
        for iz in range(2):
            x = ix + 0.5 * iz
            y = (iy + 0.5 * iz) * Nx / Ny
            pos[i, :] = np.array([x, y]) / Nx * L
            i += 1

initial_hex_positions = pos.copy() 


def run_md(T_init, steps):
   
    vel = np.random.normal(0.0, np.sqrt(T_init), size=(N, 2))
    vel -= vel.mean(axis=0)      # zero center-of-mass motion
    vel *= np.sqrt(T_init / Temp(vel, N))  # rescale
    
    pos = initial_hex_positions.copy()
    
    forces, U = compute_forces(pos)

    positions_history = []

    pos_now = pos.copy()
    v_now = vel.copy()
    f_now = forces.copy()

    for step in range(steps):
        # Velocity–Verlet integration
        v_now += 0.5 * f_now * dt
        pos_now += v_now * dt
        pos_now = apply_periodic(pos_now)
        f_new, U_new = compute_forces(pos_now)
        v_now += 0.5 * f_new * dt
        f_now = f_new

        positions_history.append(pos_now.copy())

    msd = compute_msd(positions_history, L)

    return msd

In [ ]:
temps_small = [2.0, 2.2, 2.5, 2.8, 3.0, 3.1, 3.2, 3.3, 3.4, 3.5]

msd_results_small = {}

for T in temps_small:
    print("Running T =", T)
    msd_results_small[T] = run_md(T_init=T, steps=50000)

In [ ]:
for T, msd in msd_results_small.items():
    plt.figure(figsize=(5,3))
    plt.plot(msd)
    plt.title(f"MSD at T={T}")
    plt.xlabel("Time step")
    plt.ylabel("MSD")
    plt.show()

In [ ]:
Nx = 19      # large cell example (N = 418)
Ny = 11
N = Nx * Ny * 2

L = Nx * 2**(1/6)   # choose L so that L/Nx = equilibrium LJ spacing

pos = np.zeros((N, 2))

i = 0
for ix in range(Nx):
    for iy in range(Ny):
        for iz in range(2):
            x = ix + 0.5 * iz
            y = (iy + 0.5 * iz) * Nx / Ny
            pos[i, :] = np.array([x, y]) / Nx * L
            i += 1

initial_hex_positions = pos.copy() 


def run_md(T_init, steps):
   
    vel = np.random.normal(0.0, np.sqrt(T_init), size=(N, 2))
    vel -= vel.mean(axis=0)      # zero center-of-mass motion
    vel *= np.sqrt(T_init / Temp(vel, N))  # rescale
    
    pos = initial_hex_positions.copy()
    
    forces, U = compute_forces(pos)

    positions_history = []

    pos_now = pos.copy()
    v_now = vel.copy()
    f_now = forces.copy()

    for step in range(steps):
        # Velocity–Verlet integration
        v_now += 0.5 * f_now * dt
        pos_now += v_now * dt
        pos_now = apply_periodic(pos_now)
        f_new, U_new = compute_forces(pos_now)
        v_now += 0.5 * f_new * dt
        f_now = f_new

        positions_history.append(pos_now.copy())

    msd = compute_msd(positions_history, L)

    return msd

In [ ]:
temps_large = [2.0, 2.2, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0, 3.2, 3.5]

msd_results_large = {}

for T in temps_large:
    print("Running T =", T)
    msd_results_large[T] = run_md(T_init=T, steps=10000)

In [ ]:
for T in temps_large:
    plt.figure(figsize=(5,3))
    plt.plot(msd_results_large[T])
    plt.title(f"MSD — N=418, T={T}")
    plt.xlabel("Time step")
    plt.ylabel("MSD")
    plt.show()